In [85]:
import os  # Imports Python's built-in os module.Provides functions to interact with the operating system 
from dotenv import load_dotenv  # Imports load_dotenv function from the dotenv module. This function loads environment variables from a .env file into the environment variables of the operating system.
load_dotenv() # Loads environment variables from a .env file into the operating system's environment variables.

True

In [86]:

## Langsmith Tracking and tracing
HUGGINGFACEHUB_API_TOKEN=os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="false"  # Disables LangChain tracing
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

RAG

In [87]:
from langchain_community.document_loaders import WebBaseLoader # WebBaseLoader is a class from the langchain_community.document_loaders module. It is used to load documents from a web page.
# The WebBaseLoader class is initialized with a URL, which is the web page from which to load documents.
loader=WebBaseLoader("https://python.langchain.com/docs/tutorials/llm_chain/")
loader

In [88]:
# Load the document from the specified URL
# The load method of the loader object is called to load the document from the specified URL.
document=loader.load()
document

[Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/llm_chain/', 'title': 'Build a simple LLM application with chat models and prompt templates | ğŸ¦œï¸�ğŸ”— LangChain', 'description': "In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!", 'language': 'en'}, page_content='\n\n\n\n\nBuild a simple LLM application with chat models and prompt templates | ğŸ¦œï¸�ğŸ”— LangChain\n\n\n\n\n\n\nSkip to main contentOur Building Ambient Agents with LangGraph course is now available on LangChain Academy!IntegrationsAPI ReferenceMoreContributingPeopleError referenceLangSmithLangGraphLangChain HubLangChain JS/TSv0.3v0.3v0.2v0.1ğŸ’¬SearchIntroducti

In [89]:
# Split the document into smaller chunks
# The RecursiveCharacterTextSplitter class is used to split the document into smaller chunks based on character count and overlap.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(document)
chunks

[Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/llm_chain/', 'title': 'Build a simple LLM application with chat models and prompt templates | ğŸ¦œï¸�ğŸ”— LangChain', 'description': "In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!", 'language': 'en'}, page_content='Build a simple LLM application with chat models and prompt templates | ğŸ¦œï¸�ğŸ”— LangChain'),
 Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/llm_chain/', 'title': 'Build a simple LLM application with chat models and prompt templates | ğŸ¦œï¸�ğŸ”— LangChain', 'description': "In this quickstart we'll show you how to build a simple LLM app

In [90]:
# embeddings.py
# This module provides functionality to generate embeddings for text chunks using a pre-trained model.
# It uses the SentenceTransformer library to load the model and generate embeddings for the text chunks.
from sentence_transformers import SentenceTransformer


def get_embedding_model(model_name="all-MiniLM-L6-v2"):
    print(f"[Model] Loading embedding model: {model_name}")
    return SentenceTransformer(model_name)

def generate_embeddings(chunks):
    model = get_embedding_model()

    texts = [chunk.page_content for chunk in chunks]
    metadatas = [chunk.metadata for chunk in chunks]

    print(f"[Embedding] Generating embeddings for {len(texts)} chunks...")
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    return embeddings, texts, metadatas



In [91]:
## Main execution block
# This block is executed when the script is run directly.
if __name__ == "__main__":
    
    # Embed them
    embeddings, texts, metadatas = generate_embeddings(chunks)

    # Debug print
    
    print("🔢 Embedding Vector Sample (dim 0–4):", embeddings[0][:5])
    print("📏 Total Chunks:", len(embeddings))

[Model] Loading embedding model: all-MiniLM-L6-v2
[Embedding] Generating embeddings for 25 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔢 Embedding Vector Sample (dim 0–4): [-0.0258715  -0.05476453  0.0805345  -0.10152445 -0.02972008]
📏 Total Chunks: 25


In [92]:
# load_pdfs.py
# This module provides functionality to load a PDF file and extract text from its pages.
# It uses the pdfplumber library to open the PDF and extract text from each page.   

import pdfplumber

def load_pdf(path="docs/AI-whitepaper.pdf"):
    """
    Opens the PDF and returns a list of page texts.
    """
    texts = []
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            txt = page.extract_text()
            if txt:
                texts.append(txt)
            else:
                texts.append("")  # keep page count consistent
    print(f"[load_pdfs] Extracted text from {len(texts)} pages.")
    return texts

if __name__ == "__main__":
    pages = load_pdf()
    # Optional: print first 100 characters of page 1
    print("Page 1 snippet:", pages[0][:100], "…")
    print("Total pages:", len(pages))
    print("First 10 pages:", pages[:10])


[load_pdfs] Extracted text from 174 pages.
Page 1 snippet: Agents
Authors: Julia Wiesinger, Patrick Marlow
and Vladimir Vuskovic …
Total pages: 174
First 10 pages: ['Agents\nAuthors: Julia Wiesinger, Patrick Marlow\nand Vladimir Vuskovic', 'Agents\nAcknowledgements\nContent contributors\nEvan Huang\nEmily Xue\nOlcan Sercinoglu\nSebastian Riedel\nSatinder Baveja\nAntonio Gulli\nAnant Nawalgaria\nCurators and Editors\nAntonio Gulli\nAnant Nawalgaria\nGrace Mollison\nTechnical Writer\nJoey Haymaker\nDesigner\nMichael Lanning\nFebruary 2025 2', 'Table of contents\nIntroduction 4\nWhat is an agent? 5\nThe model 6\nThe tools 7\nThe orchestration layer 7\nAgents vs. models 8\nCognitive architectures: How agents operate 8\nTools: Our keys to the outside world 12\nExtensions 13\nSample Extensions 15\nFunctions 18\nUse cases 21\nFunction sample code 24\nData stores 27\nImplementation and application 28\nTools recap 32\nEnhancing model performance with targeted learning 33\nAgent quick start with

In [93]:
## Split the loaded PDF pages into smaller chunks
# This code uses the RecursiveCharacterTextSplitter from langchain_text_splitters to split the loaded
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Suppose 'pages' is a list of strings (e.g., from pdfplumber)
documents = [Document(page_content=page) for page in pages]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"[load_pdfs] Split into {len(chunks)} chunks.")
print(chunks)

[load_pdfs] Split into 330 chunks.
[Document(metadata={}, page_content='Agents\nAuthors: Julia Wiesinger, Patrick Marlow\nand Vladimir Vuskovic'), Document(metadata={}, page_content='Agents\nAcknowledgements\nContent contributors\nEvan Huang\nEmily Xue\nOlcan Sercinoglu\nSebastian Riedel\nSatinder Baveja\nAntonio Gulli\nAnant Nawalgaria\nCurators and Editors\nAntonio Gulli\nAnant Nawalgaria\nGrace Mollison\nTechnical Writer\nJoey Haymaker\nDesigner\nMichael Lanning\nFebruary 2025 2'), Document(metadata={}, page_content='Table of contents\nIntroduction 4\nWhat is an agent? 5\nThe model 6\nThe tools 7\nThe orchestration layer 7\nAgents vs. models 8\nCognitive architectures: How agents operate 8\nTools: Our keys to the outside world 12\nExtensions 13\nSample Extensions 15\nFunctions 18\nUse cases 21\nFunction sample code 24\nData stores 27\nImplementation and application 28\nTools recap 32\nEnhancing model performance with targeted learning 33\nAgent quick start with LangChain 35\nProduct

In [94]:
# embeddings.py
# This module provides functionality to generate embeddings for text chunks using a pre-trained model.
from sentence_transformers import SentenceTransformer


def get_embedding_model(model_name="all-MiniLM-L6-v2"):
    print(f"[Model] Loading embedding model: {model_name}")
    return SentenceTransformer(model_name)

def generate_embeddings(chunks):
    model = get_embedding_model()

    texts = [chunk.page_content for chunk in chunks]
    metadatas = [chunk.metadata for chunk in chunks]

    print(f"[Embedding] Generating embeddings for {len(texts)} chunks...")
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    return embeddings, texts, metadatas



In [95]:
if __name__ == "__main__":
    
    # Embed them
    embeddings, texts, metadatas = generate_embeddings(chunks)

    # Debug print
    
    print("🔢 Embedding Vector Sample (dim 0–4):", embeddings[0][:5])
    print("📏 Total Chunks:", len(embeddings))

[Model] Loading embedding model: all-MiniLM-L6-v2
[Embedding] Generating embeddings for 330 chunks...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

🔢 Embedding Vector Sample (dim 0–4): [-0.06292843 -0.02318234 -0.05347701 -0.02287709 -0.02564664]
📏 Total Chunks: 330


In [96]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

In [97]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [98]:
# Generate embeddings for a query
# This code generates embeddings for a query using the HuggingFaceEmbeddings class.
embeddings.embed_query("What is LangChain?")

[-0.04055779054760933,
 0.013319497928023338,
 0.01940915547311306,
 0.004821693524718285,
 -0.06119084358215332,
 0.015379210002720356,
 0.05414624139666557,
 0.05520951747894287,
 0.07971914857625961,
 -0.024081269279122353,
 0.08297766745090485,
 -0.016691308468580246,
 -0.0290975458920002,
 0.02071722224354744,
 -0.06865742057561874,
 -0.0204659141600132,
 -0.07295335829257965,
 0.05014484003186226,
 0.01050212886184454,
 -0.07019736617803574,
 -0.035220205783843994,
 0.015955332666635513,
 0.046562112867832184,
 0.0327635183930397,
 0.04914434626698494,
 -0.07498354464769363,
 0.01440279558300972,
 -0.014078294858336449,
 0.0517384335398674,
 0.0038202148862183094,
 0.009939484298229218,
 0.07182702422142029,
 -0.012822557240724564,
 0.014195699244737625,
 -0.11523297429084778,
 0.12732362747192383,
 -0.008469394408166409,
 -0.007185488007962704,
 0.03286214545369148,
 0.017457954585552216,
 -0.05377723649144173,
 -0.0004547804419416934,
 0.05582141503691673,
 -0.05234828218817711

In [99]:
# Calculate cosine similarity between two embeddings
# This code calculates the cosine similarity between two embeddings using the cosine_similarity function from sklearn.metrics.p
from sklearn.metrics.pairwise import cosine_similarity


In [100]:
documents=["what is genai?","what is langchain?","what is llm?"]
query = "LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs). It provides a standardized interface for interacting with various LLMs and other tools, making it easier to build complex applications like chatbots, question-answering systems"

In [101]:
document_embedding = embeddings.embed_documents(documents)
query_embedding = embeddings.embed_query(query)

In [102]:
len(document_embedding), len(query_embedding)

(3, 384)

In [103]:
cosine_similarity(
    [query_embedding],
    document_embedding
)  # Returns a 2D array with similarity scores
# Each row corresponds to a document, and the value is the similarity score with the query. focus on angle only, not magnitude
# The higher the score, the more similar the document is to the query.

array([[0.1678599 , 0.68972841, 0.34758848]])

| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


In [104]:
# Create a FAISS vector store
# FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [105]:
# Create a FAISS index
# reffering the data dimension of the embeddings
index = faiss.IndexFlatL2(384)


In [106]:

index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000024FC15CD170> >

In [107]:
# Add embeddings to the FAISS index
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [108]:
# Add texts to the vector store
# This will create embeddings for the texts and add them to the FAISS index.
vector_store.add_texts(["AI is future","LLM agents are AI systems that leverage Large Language Models (LLMs), tools, and memory to perform tasks, make decisions, and interact with users or other systems autonomously. ","Large Language Models are a type of artificial intelligence program designed to generate human-like text and perform various natural language processing tasks. "])

['0b5066c4-c3bc-4fe9-84ee-ee183a3ab0f4',
 '488c2a44-56e6-48c1-b52c-57a652a0b852',
 '8810b37a-a597-401c-af5e-ea17175bc69a']

In [109]:
# Check the mapping of index to docstore IDs
# This will show the mapping of the index positions to the document IDs in the docstore.
vector_store.index_to_docstore_id

{0: '0b5066c4-c3bc-4fe9-84ee-ee183a3ab0f4',
 1: '488c2a44-56e6-48c1-b52c-57a652a0b852',
 2: '8810b37a-a597-401c-af5e-ea17175bc69a'}

In [110]:
# Perform a similarity search
results = vector_store.similarity_search("Tell me about AI", k=3)

In [111]:
results

[Document(id='0b5066c4-c3bc-4fe9-84ee-ee183a3ab0f4', metadata={}, page_content='AI is future'),
 Document(id='488c2a44-56e6-48c1-b52c-57a652a0b852', metadata={}, page_content='LLM agents are AI systems that leverage Large Language Models (LLMs), tools, and memory to perform tasks, make decisions, and interact with users or other systems autonomously. '),
 Document(id='8810b37a-a597-401c-af5e-ea17175bc69a', metadata={}, page_content='Large Language Models are a type of artificial intelligence program designed to generate human-like text and perform various natural language processing tasks. ')]

In [112]:
| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search        | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed                 | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


SyntaxError: invalid syntax (3303128725.py, line 1)

In [ ]:
# Flat Index
# - Type of Search: Exact       
# - Speed: Slow (linear scan)
# - Memory Usage: High (stores all vectors)
# - Use Case: Small datasets where exact matches are needed
# - Example: Used in applications like small-scale search engines, recommendation systems, and personal assistants

# IVF Index
# - Type of Search: Approximate (cluster-based) 
# - Speed: Fast (search only in top clusters)
# - Memory Usage: Moderate (stores cluster centroids)
# - Use Case: Medium datasets where speed is more important than exact matches
# - Example: Used in applications like image retrieval, document search, and recommendation systems


# HNSW Index
# - Type of Search: Approximate (graph-based traversal)
# - Speed: Very Fast (graph walk)
# - Memory Usage: Low (stores graph structure)
# - Use Case: Large datasets where speed is critical and approximate matches are acceptable
# - Example: Used in applications like image search, recommendation systems, and large-scale text retrieval
# - Note: HNSW is often preferred for large datasets due to its speed and efficiency in finding nearest neighbors.

In [ ]:
| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                   | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                   | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


In [ ]:
# from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
# Create a FAISS index
# Using Inner Product (IP) for similarity search cosine similarity
# The FAISS index is created with the dimension of the embeddings (384 in this case).
index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
vector_store.add_documents(documents=documents)

['b236b36f-b3e6-47a1-b886-ee3e3114920a',
 '48142cc6-941a-4633-883d-dd8349555bfe',
 'baaf6a59-2861-429d-b681-2133aa5fded5',
 '6f5aee8b-ac69-4b35-b5e4-2e99b5376643',
 '867d7000-a5a4-4d17-805f-affd6e156dac',
 'b5a6e719-0b90-4b46-9dcc-f3c8161cf760',
 '7c81fd04-0f17-4b2d-a943-6347ce8f1371',
 '9bd1f382-4f85-4d2e-893f-d7f189c6b9af',
 'bdf5b24b-a23c-424c-9ad0-9f6c81b03dc8',
 'd02f42ea-cb97-4f12-8004-0aaebb3e7474']

In [ ]:
# Perform a similarity search
# This will return the top k most similar documents to the query based on cosine similarity.
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=6, #hyperparameter 
    #filter={"source":{"$eq": "tweet"}}
    
)

[Document(id='baaf6a59-2861-429d-b681-2133aa5fded5', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='9bd1f382-4f85-4d2e-893f-d7f189c6b9af', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='d02f42ea-cb97-4f12-8004-0aaebb3e7474', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='b236b36f-b3e6-47a1-b886-ee3e3114920a', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(id='867d7000-a5a4-4d17-805f-affd6e156dac', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again.")]

In [ ]:
result=vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    #k=2 #hyperparameter,
    filter={"source":"news"}
    
)
result

[Document(id='6f5aee8b-ac69-4b35-b5e4-2e99b5376643', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='48142cc6-941a-4633-883d-dd8349555bfe', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='bdf5b24b-a23c-424c-9ad0-9f6c81b03dc8', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]

In [ ]:
# Accessing metadata of the first result
# This will return the metadata of the first result in the similarity search.
result[0].metadata

{'source': 'news'}

In [ ]:
# Accessing the page content of the first result
result[0].page_content

'Robbers broke into the city bank and stole $1 million in cash.'

In [ ]:
retriever=vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019830C329F0>, search_kwargs={'k': 3})

In [ ]:
retriever.invoke("LangChain provides abstractions to make working with LLMs easy")

[Document(id='baaf6a59-2861-429d-b681-2133aa5fded5', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='9bd1f382-4f85-4d2e-893f-d7f189c6b9af', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='d02f42ea-cb97-4f12-8004-0aaebb3e7474', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

inmemory(server)
ondisk(server)
cloud(-----)

In [ ]:
vector_store.save_local("faiss_index_data")

In [ ]:
# Loading the FAISS index from local storage
# This will load the FAISS index from the specified directory and allow dangerous deserialization.
# allow_dangerous_deserialization=True is used to allow loading of potentially unsafe data.
# This is useful when you are sure that the data being loaded is safe and you want to avoid potential security issues that may arise from loading untrusted data.
new_vector_store = FAISS.load_local("faiss_index_data", embeddings,allow_dangerous_deserialization=True)  

In [ ]:
new_vector_store.similarity_search("langchain")

[Document(id='baaf6a59-2861-429d-b681-2133aa5fded5', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='9bd1f382-4f85-4d2e-893f-d7f189c6b9af', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='867d7000-a5a4-4d17-805f-affd6e156dac', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(id='7c81fd04-0f17-4b2d-a943-6347ce8f1371', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
FILE_PATH=r"V:\KAI\call_llm\llm_workspace\Langchain\docs\AI-whitepaper.pdf"

In [ ]:
#
loader = PyPDFLoader(FILE_PATH)
pages = loader.load()
pages

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-07-02T08:57:25+00:00', 'source': 'V:\\KAI\\call_llm\\llm_workspace\\Langchain\\docs\\AI-whitepaper.pdf', 'total_pages': 174, 'page': 0, 'page_label': '1'}, page_content='Agents\nAuthors: Julia Wiesinger, Patrick Marlow  \nand Vladimir Vuskovic'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-07-02T08:57:25+00:00', 'source': 'V:\\KAI\\call_llm\\llm_workspace\\Langchain\\docs\\AI-whitepaper.pdf', 'total_pages': 174, 'page': 1, 'page_label': '2'}, page_content='Agents\n2\nFebruary 2025\nAcknowledgements\nContent contributors\nEvan Huang\nEmily Xue\nOlcan Sercinoglu\nSebastian Riedel\nSatinder Baveja\nAntonio Gulli\nAnant Nawalgaria\nCurators and Editors\nAntonio Gulli\nAnant Nawalgaria\nGrace Mollison \nTechnical Writer\nJoey Haymaker\nDesigner\nMichael Lanning'),
 Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creatio

In [ ]:
# This will load the PDF pages lazily, meaning it will not load all pages into memory at once.
# This is useful for large PDFs where you want to load pages on demand rather than loading the entire PDF into memory at once.
# Lazy loading can help improve performance and reduce memory usage when working with large documents. 
len(pages)  # This will return the number of pages in the PDF document.
pages = []
async for page in loader.alazy_load():
    pages.append(page)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,#hyperparameter
    chunk_overlap=50 #hyperparemeter
)

In [ ]:
split_docs = splitter.split_documents(pages)

In [ ]:
len(split_docs)  # This will return the number of chunks created from the PDF pages.

566

In [ ]:
# Create a FAISS index
index=faiss.IndexFlatIP(566)  # 566 is the dimension of the embeddings for the text chunks
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
# Add the split documents to the vector store
# This will create embeddings for the text chunks and add them to the FAISS index.
vector_store.add_documents(documents=split_docs)

['4ece7d10-a85e-461d-9e13-361401789d66',
 'ffb3cdf7-c0c0-41eb-b16f-058cca54c953',
 '697fcfae-eb26-48b5-b47e-0ee697f68f1a',
 'd4b1e85b-ab48-4a2b-9b67-6cfdb1e660d2',
 '8590148a-39ef-43bc-a41f-decfd7e04a7d',
 'd0984b76-7c0b-4a98-895b-548c7ebbd599',
 '77e0b550-5110-4e89-be53-2f739fb4e1cf',
 '473ec4ec-7055-4d9b-bea6-23710032e8dd',
 '730689c3-5161-41b4-96cc-0367d314d437',
 'c08fbdae-1e38-41c5-88f0-b99b6120fa97',
 '201ed9ea-9d1d-45c4-8fd6-061fb1875603',
 '46ed7b34-ecec-4386-93f7-8f52c5c661bf',
 'b81405a1-5346-46b6-b7ff-08a73dc0f308',
 '1e11757d-9782-47dd-a255-58d92bd07ebd',
 '4833b4c4-21fd-469d-849e-00513d407be8',
 '564462f2-5632-453e-922e-be84ecbc9419',
 '80d5ba53-c6d2-419b-b47b-ffae4f48a45e',
 '97b843f4-2134-479b-9dd0-74f442164ff2',
 '6ea17f54-1ddd-4afb-acff-e9758fccbb94',
 '5eb66ca0-a81e-4249-a1e1-c86a264604b0',
 '3046691f-b1ee-4602-8d9b-0e34f3be51a4',
 '76fa8004-7b3f-4fdb-a816-fa6a1134495a',
 '50adaf81-67b7-42fb-ad20-139727e5a71d',
 '176a4f04-7be2-4ff0-a2c6-74d40bc7a9b4',
 '79ef0f72-d13b-

In [ ]:
# Create a retriever from the vector store
# This will allow you to perform similarity searches on the vector store.
retriever=vector_store.as_retriever(
    search_kwargs={"k": 10} #hyperparameter
)

In [ ]:
retriever.invoke("what is Agent?")


RunTree.patch() got an unexpected keyword argument 'exclude_inputs'
Error in LangChainTracer.on_retriever_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")


[Document(id='4329fd03-019a-40bc-b584-02c7cfbe83f5', metadata={}, page_content='robots are autonomus'),
 Document(id='a7ab0bb5-9584-42e5-95f6-9adf5c9a136d', metadata={}, page_content='AI is future'),
 Document(id='fb77be9c-e2b1-4182-b6cc-f661807ad757', metadata={}, page_content='AI is the next big change')]

In [ ]:
from langchain.llms import Ollama # Imports the Ollama class from the langchain.llms module. This class is used to interact with the Ollama LLMs (Large Language Models).

# Load the Qwen 0.5B model
model = Ollama(model="qwen:0.5b")

In [ ]:
# Load the RAG prompt from the LangChain Hub
# The hub.pull function is used to pull the prompt from the LangChain Hub repository.
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
## Create a chain that combines the retriever and the LLM with the RAG prompt
import pprint
# The pprint module is used to pretty-print the prompt messages in a more readable format.
# This will display the messages in the prompt in a structured format.
pprint.pprint(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [ ]:

from langchain_core.output_parsers import StrOutputParser # Imports the StrOutputParser class from the langchain_core.output_parsers module. This class is used to parse the output of the LLM into a string format.
from langchain_core.runnables import RunnablePassthrough # RunnablePassthrough is a class that allows you to pass through the input without any modifications.

context(retriever),prompt(hub),model(Ollama),parser(langchain)

In [ ]:

def format_docs(docs): # This function formats the documents by joining their page content with two newlines.
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [114]:
rag_chain.invoke("what are large language model ?")

Error in LangChainTracer.on_chain_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")


Error in LangChainTracer.on_retriever_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")
Error in LangChainTracer.on_chain_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")
Error in LangChainTracer.on_chain_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")
Error in LangChainTracer.on_chain_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")
Error in LangChainTracer.on_chain_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")
Error in LangChainTracer.on_llm_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")
Error in LangChainTracer.on_chain_end callback: TypeError("RunTree.patch() got an unexpected keyword argument 'exclude_inputs'")
Error in LangChainTracer.on_chain_end callback: TypeError("RunTree.patch() got an unexpected ke

'A large language model is a type of artificial intelligence (IA) that is designed to process large amounts of text data.\n\nThese models can be trained on various sources, such as books, articles, and websites. Once trained, these models can be used for a variety of tasks, including language translation, sentiment analysis, topic modeling, and more.\n\nOverall, large language models are highly advanced artificial intelligence systems that have the potential to revolutionize many fields.'